# Phase 2: MAPPO robustness, transfer, and reports
Each experiment section can run after initialization in a fresh runtime. A fresh, post-repair Phase-1 baseline run and nominal IPPO run are mandatory: the gate below stops execution unless every safeguard passes. Never interpret Pymunk transfer as physical deployment.

## 1. Compact initialization

In [1]:
from pathlib import Path
import json, os, sys
REPO_URL = "https://github.com/djdhillxn/football"
REPO_DIR = Path("/content/robot-soccer-transfer")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "RobotSoccerTransfer"
!python --version
!nvidia-smi || true
from google.colab import drive
drive.mount(str(DRIVE_MOUNT))
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
os.environ["ROBOSOCCER_DRIVE_PROJECT"] = str(DRIVE_PROJECT)
if REPO_DIR.exists():
    dirty = !git -C {REPO_DIR} status --porcelain
    if dirty:
        raise RuntimeError("Repository has local changes; refusing to overwrite them.")
    !git -C {REPO_DIR} fetch origin --quiet
    !git -C {REPO_DIR} pull --ff-only
else:
    if REPO_URL.startswith("REPLACE_"):
        raise RuntimeError("Set REPO_URL before cloning.")
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git rev-parse HEAD
# Preserve Colab's CUDA-enabled Torch, NumPy, and pandas builds.
!python -m pip install -q gymnasium==1.2.1 pettingzoo==1.26.1 pymunk==7.3.0 PyYAML==6.0.3 tensorboard==2.20.0 imageio==2.37.2 imageio-ffmpeg==0.6.0 pytest==8.4.2 ruff==0.14.5
if int(get_ipython().user_ns.get("_exit_code", 0)) != 0:
    raise RuntimeError("dependency installation failed")
!python -m pip install -e . --no-deps -q
if int(get_ipython().user_ns.get("_exit_code", 0)) != 0:
    raise RuntimeError("editable package installation failed")
sys.path.insert(0, str(REPO_DIR))
import torch, robosoccer
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "project", robosoccer.__version__)

Mounted at /content/drive
Cloning into '/content/robot-soccer-transfer'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 72 (delta 17), reused 69 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 105.87 KiB | 272.00 KiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/robot-soccer-transfer
16c94cd4ff55ba014fc5865b32a0f2ab380a0110
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 951.1/951.1 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.5/805.5 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 147.2 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to bui

In [2]:
def require_shell_success(label):
    status = int(get_ipython().user_ns.get("_exit_code", 0))
    if status != 0:
        raise RuntimeError(f"{label} failed with shell status {status}")

def latest_run(name):
    pointer = REPO_DIR / "runs" / f"latest_{name}.txt"
    if not pointer.is_file():
        raise FileNotFoundError(pointer)
    run_dir = Path(pointer.read_text().strip())
    return run_dir if run_dir.is_absolute() else (REPO_DIR / run_dir).resolve()

def display_json(path):
    data = json.loads(Path(path).read_text())
    print(json.dumps(data, indent=2)[:5000])
    return data

from robosoccer.config import load_config
from robosoccer.evaluation import phase1_readiness_audit
from robosoccer.artifacts import (
    sync_artifacts_from_drive,
    sync_auxiliary_artifacts_to_drive,
    sync_run_to_drive,
    sync_workspace_to_drive,
)

def local_run_names():
    runs_root = REPO_DIR / "runs"
    return {path.name for path in runs_root.iterdir() if path.is_dir()} if runs_root.is_dir() else set()

def sync_training_result(previous_runs, label):
    status = int(get_ipython().user_ns.get("_exit_code", 0))
    new_runs = sorted(local_run_names() - set(previous_runs))
    finished = []
    for name in new_runs:
        run_dir = REPO_DIR / "runs" / name
        metadata_path = run_dir / "run_metadata.json"
        if metadata_path.is_file():
            metadata = json.loads(metadata_path.read_text())
            if metadata.get("status") in {"complete", "failed"}:
                finished.append(run_dir)
    if not finished:
        if status != 0:
            raise RuntimeError(f"{label} failed with shell status {status}; no finished run was created.")
        raise RuntimeError(f"{label} completed without creating a finished run directory.")
    run_dir = finished[-1]
    print(json.dumps(sync_run_to_drive(run_dir, DRIVE_PROJECT), indent=2))
    if status != 0:
        raise RuntimeError(f"{label} failed with shell status {status}; diagnostics were saved to Drive.")
    return run_dir

def sync_run_after_command(run_dir, label):
    status = int(get_ipython().user_ns.get("_exit_code", 0))
    print(json.dumps(sync_run_to_drive(run_dir, DRIVE_PROJECT), indent=2))
    if status != 0:
        raise RuntimeError(f"{label} failed with shell status {status}; diagnostics were saved to Drive.")
    return Path(run_dir)

def sync_auxiliary_after_command(label, include_reports=True):
    status = int(get_ipython().user_ns.get("_exit_code", 0))
    result = sync_auxiliary_artifacts_to_drive(DRIVE_PROJECT, REPO_DIR, include_reports)
    print(json.dumps(result, indent=2))
    if status != 0:
        raise RuntimeError(f"{label} failed with shell status {status}; partial artifacts were saved.")
    return result

def require_phase1_go():
    baseline_run = latest_run("base")
    ippo_run = latest_run("ippo_nominal")
    baseline_summary = json.loads((baseline_run / "eval" / "baseline_summary.json").read_text())
    abstract_summary = json.loads((ippo_run / "eval" / "abstract_standard" / "summary.json").read_text())
    transfer_summary = json.loads((ippo_run / "eval" / "pymunk_transfer" / "summary.json").read_text())
    audit = phase1_readiness_audit(
        load_config(REPO_DIR / "configs" / "base.yaml"),
        baseline_summary,
        abstract_summary,
        transfer_summary,
    )
    print(json.dumps(audit, indent=2))
    if not audit["phase2_ready"]:
        raise RuntimeError("PHASE 2 NO-GO: rerun or repair Phase 1 before training MAPPO.")
    print("PHASE 2 GO: all Phase-1 safeguards passed.")
    return audit

In [4]:
# Restore every finished run and generated report artifact into this fresh runtime.
drive_sync = sync_artifacts_from_drive(
    DRIVE_PROJECT, REPO_DIR, include_training_artifacts=True
)
print(json.dumps(drive_sync, indent=2))

{
  "runs": "/content/robot-soccer-transfer/runs",
  "reports": "/content/robot-soccer-transfer/reports",
  "comparisons": "not present on Drive"
}
Restored 3 latest-run pointers.


## 2. Restore the required Phase-1 artifacts and enforce the gate

In [5]:
# The preceding full pull restored Phase-1 artifacts and rebuilt local pointers.
phase1_audit = require_phase1_go()

{
  "baseline_ready": true,
  "phase2_ready": true,
  "checks": {
    "bounded_fraction_metrics": {
      "passed": true,
      "observed": true,
      "criterion": "every reported fraction is in [0, 1]"
    },
    "baseline_evaluation_complete": {
      "passed": true,
      "observed": {
        "random__abstract": 100,
        "random__pymunk": 100,
        "double_chase__abstract": 100,
        "double_chase__pymunk": 100,
        "role_based__abstract": 100,
        "role_based__pymunk": 100
      },
      "criterion": "each scripted method/simulator cell has at least 100 episodes"
    },
    "pymunk_random_not_saturated": {
      "passed": true,
      "observed": 0.49,
      "criterion": "random Pymunk success <= 0.5"
    },
    "pymunk_policy_sensitive": {
      "passed": true,
      "observed": 0.19000000000000006,
      "criterion": "Pymunk scripted-method success spread >= 0.15"
    },
    "role_success_exceeds_random": {
      "passed": true,
      "observed": {
        "abs

## 3. Train nominal MAPPO
This section is reachable only after the Phase-1 audit reports `phase2_ready: true`.

In [4]:
previous_runs = local_run_names()
!python -m scripts.train --config configs/mappo_nominal.yaml
nominal_run = sync_training_result(previous_runs, "nominal MAPPO training")

2026-07-21 11:34:08.166443: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 11:34:08.237527: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-07-21 11:34:10,032:jax._src.path:41: etils.epath found. Using etils.epath for file I/O.
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

  parse = parser.parseString(pattern)

  parser.resetCache()

  ParserElement.enablePackrat()

INFO: Training MAPPO with 32 environments for 2,000,000 steps.
MAPPO:   0% 0/245 [00:00<?, ?it/

## 4. Train uniformly randomized MAPPO

In [5]:
previous_runs = local_run_names()
!python -m scripts.train --config configs/mappo_uniform_dr.yaml
uniform_run = sync_training_result(previous_runs, "uniform-randomization MAPPO training")

2026-07-21 13:06:58.991576: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 13:06:59.064019: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-07-21 13:07:00,874:jax._src.path:41: etils.epath found. Using etils.epath for file I/O.
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

  parse = parser.parseString(pattern)

  parser.resetCache()

  ParserElement.enablePackrat()

INFO: Training MAPPO with 32 environments for 2,000,000 steps.
MAPPO:   0% 0/245 [00:00<?, ?it/

## 5. Train failure-directed MAPPO

In [6]:
previous_runs = local_run_names()
!python -m scripts.train --config configs/mappo_failure_dr.yaml
failure_run = sync_training_result(previous_runs, "failure-directed MAPPO training")

2026-07-21 19:44:38.862526: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 19:44:38.931767: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:2026-07-21 19:44:40,647:jax._src.path:41: etils.epath found. Using etils.epath for file I/O.
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

  parse = parser.parseString(pattern)

  parser.resetCache()

  ParserElement.enablePackrat()

INFO: Training MAPPO with 32 environments for 2,000,000 steps.
MAPPO:   0% 0/245 [00:00<?, ?it/

: 

## 6. Optional focused ablations

In [ ]:
# Run only after the primary methods and only if the experiment budget allows.
previous_runs = local_run_names()
!python -m scripts.train --config configs/mappo_failure_dr.yaml --run-name mappo_failure_no_action_delay 'randomization.disabled_families=[action_delay]'
delay_ablation_run = sync_training_result(previous_runs, "action-delay ablation")
previous_runs = local_run_names()
!python -m scripts.train --config configs/mappo_failure_dr.yaml --run-name mappo_failure_no_communication 'randomization.disabled_families=[communication]'
communication_ablation_run = sync_training_result(previous_runs, "communication ablation")
# Optional: replace the family below with observation_noise.
# previous_runs = local_run_names()
#!python -m scripts.train --config configs/mappo_failure_dr.yaml --run-name mappo_failure_no_observation_noise 'randomization.disabled_families=[observation_noise]'
# observation_ablation_run = sync_training_result(previous_runs, "observation-noise ablation")

## 7. Evaluate learned policies in the abstract simulator

In [ ]:
for experiment in ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]:
    run_dir = latest_run(experiment)
    !python -m scripts.evaluate --run-dir {run_dir} --checkpoint best --simulator abstract --suite standard
    sync_run_after_command(run_dir, f"{experiment} abstract evaluation")

## 8. Evaluate the same frozen actors in Pymunk

In [ ]:
for experiment in ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]:
    run_dir = latest_run(experiment)
    !python -m scripts.evaluate --run-dir {run_dir} --checkpoint best --simulator pymunk --suite transfer
    sync_run_after_command(run_dir, f"{experiment} Pymunk transfer evaluation")

## 9. Run profile evaluations

In [ ]:
for experiment in ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]:
    run_dir = latest_run(experiment)
    !python -m scripts.evaluate --run-dir {run_dir} --checkpoint best --simulator pymunk --suite profiles
    sync_run_after_command(run_dir, f"{experiment} profile evaluation")

## 10. Run robustness grids

In [ ]:
for experiment in ["mappo_nominal", "mappo_uniform_dr", "mappo_failure_dr"]:
    run_dir = latest_run(experiment)
    !python -m scripts.evaluate --run-dir {run_dir} --checkpoint best --simulator pymunk --suite robustness
    sync_run_after_command(run_dir, f"{experiment} robustness grid")

## 11. Record nominal and transfer videos

In [ ]:
failure_run = latest_run("mappo_failure_dr")
!python -m scripts.record_video --run-dir {failure_run} --checkpoint best --simulator abstract --episodes 3
sync_run_after_command(failure_run, "failure-directed abstract videos")
!python -m scripts.record_video --run-dir {failure_run} --checkpoint best --simulator pymunk --episodes 3
sync_run_after_command(failure_run, "failure-directed transfer videos")

## 12. Compare all completed runs

In [ ]:
!python -m scripts.compare_runs --phase final --export-report
sync_auxiliary_after_command("final run comparison")

## 13. Display the main CSV and selected figures

In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage
comparison_dir = sorted((REPO_DIR / "runs" / "comparisons").glob("*_final"))[-1]
display(pd.read_csv(comparison_dir / "main_comparison.csv"))
for figure in [comparison_dir / "method_success_comparison.png", comparison_dir / "transfer_gap.png"]:
    if figure.is_file():
        display(DisplayImage(filename=str(figure)))

## 14. Build the main paper-style report

In [ ]:
!latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/main.tex
sync_auxiliary_after_command("main report build")

## 15. Build the detailed surrogate report

In [ ]:
!latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/surrogate_notes.tex
sync_auxiliary_after_command("surrogate report build")

## 16. Sync runs, reports, PDFs, plots, and videos to Drive

In [ ]:
# Safety net only; every expensive section above already saves immediately.
print(json.dumps(sync_workspace_to_drive(DRIVE_PROJECT, REPO_DIR), indent=2))

## 17. Finished
Confirm the Drive copies and compiled PDFs, record failed or excluded runs in `surrogate_notes.tex`, and disconnect the Colab runtime.